# Multimodal RAG Ingestion — End-to-End Walkthrough

This notebook reconstructs the workflow implemented across four linked modules:

1. `loaders.py` — discovers charts, tables, and PDFs and converts them into LangChain `Document` objects.
2. `captioner.py` — converts images into searchable text captions using a vision-language model.
3. `ingest.py` — orchestrates loading → captioning → filtering → embedding → FAISS persistence.
4. `ingest_multimodal.py` — CLI entry point that exposes the ingestion pipeline and performs a retrieval smoke test.

The objective is not just to run the code, but to understand the **input/output contract at every stage**, what changes inside a `Document`, and how the final FAISS vector store is produced.

> **Important:** The cells that call the vision API or build the real FAISS index are marked as execution cells that require the original project environment, credentials, and corpus. The inspection/demo cells are safe to run independently once the project is importable.


## 1. Overall architecture

```text
documents/multimodal/
│
├── charts/                  ─┐
├── tables/                  ─┼──► loaders.py ──► List[Document]
├── pdfs/                    ─┤                         │
└── metadata/                ─┘                         │
                                                       ▼
                                             needs_caption?
                                               /        \
                                             YES        NO
                                              │          │
                                              ▼          │
                                        captioner.py     │
                                        image → text     │
                                              │          │
                                              └────┬─────┘
                                                   ▼
                                            usable Documents
                                                   │
                                                   ▼
                                         text embeddings
                                         text-embedding-3-small
                                                   │
                                                   ▼
                                            FAISS vectorstore
                                                   │
                                                   ▼
                                  data/vectorstore/faiss_multimodal/
                                                   │
                                                   ▼
                                        similarity_search()
```

### Design principle

The implementation deliberately uses **caption-then-embed** rather than directly embedding images. The vision model is used at ingestion time; after captioning, the downstream pipeline remains text-based and can reuse the same text embedding strategy used by the existing RAG system.


## 2. The four modules and their responsibilities

| Module | Main responsibility | Important input | Important output |
|---|---|---|---|
| `loaders.py` | Corpus discovery + `Document` construction | Files + metadata JSON | `list[Document]` |
| `captioner.py` | Image → structured text | Image path + optional context | `str` caption |
| `ingest.py` | End-to-end orchestration | Corpus directory + worker/cache settings | `FAISS` |
| `ingest_multimodal.py` | CLI / operational entry point | CLI arguments | Ingestion run + retrieval smoke test |

The separation is intentional: loading has no API dependency, captioning has one focused responsibility, and ingestion owns orchestration.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Notebook setup
from pathlib import Path
import json
import os
import sys
from collections import Counter

# Adjust this to the root of the agentic_bi_platform project.
PROJECT_ROOT = Path.cwd().parents[0]
SRC_DIR = PROJECT_ROOT / "src"

if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src exists:", SRC_DIR.exists())


PROJECT_ROOT: /Users/azizulshaikh/Projects/agentic_bi_platform
src exists: True


## 3. `loaders.py`: the first transformation

The loader converts heterogeneous files into a common abstraction:

```python
Document(
    page_content=<text>,
    metadata={<metadata fields>}
)
```

This is the key normalization step.

### `Document.page_content`

- For chart/table images: initially the metadata `key_insight` is used as a **placeholder**.
- For PDF text pages: extracted PDF text is placed here immediately.
- For PDF embedded images: it starts empty and is filled by the captioning stage.

### `Document.metadata`

The most important fields are:

- `source_type`: `chart`, `table`, `pdf_text`, or `pdf_image`
- `file_path`: absolute source path
- `file_name`: source filename
- `title`
- `key_insight`
- `page_number` for PDF content
- `image_index` for PDF images
- `needs_caption`: controls whether the vision model is required

This `needs_caption` flag is the hand-off contract between `loaders.py` and `ingest.py`.


In [3]:
# Inspect the loader implementation from the uploaded source.
loader_source = Path(SRC_DIR / "agentic_bi/rag/multimodal/loaders.py").read_text()
print(loader_source[:5000])


"""
loaders.py
──────────
Converts the three multimodal artefact types into lists of
langchain_core.documents.Document objects ready for captioning and
embedding.

Each Document carries:
  - page_content: the text that will be embedded (caption for images,
                  extracted text for PDF text pages)
  - metadata dict with:
      source_type:   "chart" | "table" | "pdf_text" | "pdf_image"
      file_path:     absolute path to the original file (str)
      file_name:     filename only, e.g. "01_monthly_revenue_trend.png"
      title:         human-readable title from metadata JSON (if available)
      key_insight:   pre-written insight from metadata JSON (if available)
      page_number:   page index (PDFs only)
      image_index:   position of image within PDF page (pdf_image only)
      needs_caption: True  → page_content is a placeholder; captioner
                             must replace it before embedding
                    False → page_content is already usable text

Tw

## 4. Expected corpus layout

The loader expects this structure:

```text
multimodal/
├── charts/
│   ├── *.png
│   ├── *.jpg
│   └── ...
├── tables/
│   └── *.png
├── pdfs/
│   └── *.pdf
└── metadata/
    ├── charts_metadata.json
    ├── tables_metadata.json
    └── pdfs_metadata.json
```

Each metadata JSON is treated as a lookup keyed by filename.

For an image, conceptually:

```text
filename
   │
   ├──► image file
   │
   └──► metadata_lookup[filename]
             │
             ├── title
             ├── key_insight
             ├── chart_type
             ├── time_period
             └── data_source
```


In [4]:
# Optional: inspect the real corpus if it exists in this environment.
# Change CORPUS_DIR if your project uses a different location.

CORPUS_DIR = PROJECT_ROOT / "documents" / "multimodal"

if CORPUS_DIR.exists():
    for p in sorted(CORPUS_DIR.rglob("*")):
        if p.is_file():
            print(p.relative_to(CORPUS_DIR))
else:
    print("Corpus directory not found:", CORPUS_DIR)
    print("Set CORPUS_DIR to your actual documents/multimodal path.")


charts/01_monthly_revenue_trend.png
charts/02_quarterly_revenue_bar.png
charts/03_category_revenue_bar.jpg
charts/04_category_return_rate.png
charts/05_regional_revenue_pie.png
charts/06_regional_delivery_days.png
charts/07_payment_method_donut.jpg
charts/08_review_score_distribution.png
charts/09_customer_segment_revenue.png
charts/10_aov_vs_delivery.png
metadata/charts_metadata.json
metadata/pdfs_metadata.json
metadata/tables_metadata.json
pdfs/P01_q2_2018_business_report.pdf
pdfs/P02_revenue_decline_analysis.pdf
pdfs/P03_product_category_deep_dive.pdf
pdfs/_extracted/P01_q2_2018_business_report_page2_img0.png
pdfs/_extracted/P01_q2_2018_business_report_page2_img1.png
pdfs/_extracted/P01_q2_2018_business_report_page3_img0.png
pdfs/_extracted/P01_q2_2018_business_report_page4_img0.png
pdfs/_extracted/P01_q2_2018_business_report_page4_img1.png
pdfs/_extracted/P01_q2_2018_business_report_page5_img0.png
pdfs/_extracted/P01_q2_2018_business_report_page5_img1.png
pdfs/_extracted/P02_revenu

## 5. Image loading: `load_image_documents()`

For every supported image (`png`, `jpg`, `jpeg`, `webp`), the loader creates **one LangChain `Document`**.

Initial state:

```text
page_content = key_insight or ""
needs_caption = True
```

That means the `key_insight` is not considered the final visual description. It acts as useful context for the later vision call.

### Input → output contract

**Input**

```python
image_dir: Path
source_type: str
metadata_lookup: dict
```

**Output**

```python
list[Document]
```

Each output document has image-specific metadata and is marked `needs_caption=True`.


## 6. PDF loading is intentionally two-track

`load_pdf_documents()` handles two different knowledge channels inside a PDF:

### A. PDF text

```text
PDF page
  └── extract_text()
       └── if length >= min_text_length
             └── Document(
                    page_content=<extracted text>,
                    needs_caption=False
                )
```

### B. Embedded images

```text
PDF page
  └── page.images
       └── save image to _extracted/
             └── Document(
                    page_content="",
                    needs_caption=True
                )
```

This prevents visual information from being lost merely because the PDF also contains text.

The default text threshold is **80 characters**. Pages below that threshold are not added as PDF text documents.


## 7. Inspect actual `Document` objects

The following cell uses the project's loader directly when the project environment and corpus are available.


In [5]:
# Real loader execution — requires the project dependencies and corpus.
try:
    from agentic_bi.rag.multimodal.loaders import load_multimodal_corpus

    docs = load_multimodal_corpus(CORPUS_DIR)
    print("Total Documents:", len(docs))

    for i, doc in enumerate(docs[:5]):
        print(f"\n--- Document {i} ---")
        print("page_content type :", type(doc.page_content).__name__)
        print("page_content chars:", len(doc.page_content or ""))
        print("metadata keys     :", list(doc.metadata.keys()))
        print("source_type       :", doc.metadata.get("source_type"))
        print("file_name         :", doc.metadata.get("file_name"))
        print("needs_caption     :", doc.metadata.get("needs_caption"))
except Exception as e:
    print("Loader execution skipped/failed:", type(e).__name__, "-", e)
    print("This notebook can still be used as the conceptual walkthrough.")


Total Documents: 40

--- Document 0 ---
page_content type : str
page_content chars: 213
metadata keys     : ['source_type', 'file_path', 'file_name', 'title', 'key_insight', 'chart_type', 'time_period', 'data_source', 'needs_caption']
source_type       : chart
file_name         : 01_monthly_revenue_trend.png
needs_caption     : True

--- Document 1 ---
page_content type : str
page_content chars: 197
metadata keys     : ['source_type', 'file_path', 'file_name', 'title', 'key_insight', 'chart_type', 'time_period', 'data_source', 'needs_caption']
source_type       : chart
file_name         : 02_quarterly_revenue_bar.png
needs_caption     : True

--- Document 2 ---
page_content type : str
page_content chars: 250
metadata keys     : ['source_type', 'file_path', 'file_name', 'title', 'key_insight', 'chart_type', 'time_period', 'data_source', 'needs_caption']
source_type       : chart
file_name         : 03_category_revenue_bar.jpg
needs_caption     : True

--- Document 3 ---
page_content typ

## 8. `needs_caption` creates the branching point

After loading:

```python
to_caption = [d for d in all_docs if _should_caption(d)]
text_ready = [d for d in all_docs if not _should_caption(d)]
```

So the corpus is split into:

```text
                    all_docs
                       │
              ┌────────┴────────┐
              ▼                 ▼
       needs caption        text already ready
              │                 │
              ▼                 │
        vision model            │
              │                 │
              └────────┬────────┘
                       ▼
                  all_ready
```

`_should_caption()` also checks the image size. Embedded images smaller than **5,000 bytes** are skipped because they are assumed to be unlikely to contain meaningful analytical content.


# Part II — Captioning

## 9. `captioner.py`: image → text

The captioner has a deliberately narrow responsibility.

```text
Image file
   │
   ▼
image_to_base64()
   │
   ├── base64 string
   └── media type
   │
   ▼
HumanMessage
   ├── image_url = data:<media_type>;base64,<data>
   └── text prompt
   │
   ▼
Vision LLM
   │
   ▼
response.content
   │
   ▼
plain-text caption
```

The caption is intended to become `Document.page_content`, which means it is subsequently embedded like normal text.


In [6]:
captioner_source = Path(SRC_DIR / "agentic_bi/rag/multimodal/captioner.py").read_text()
print(captioner_source[:6500])

"""
captioner.py
────────────
Converts image files to rich text captions via a vision-language model.

Why caption-then-embed rather than direct joint embedding?
──────────────────────────────────────────────────────────
Direct joint embedding (CLIP-style) requires a separate multimodal
embedding model that maps images and text into one shared vector space.
That adds a new model dependency, a different embedding dimension than
text-embedding-3-small, and forces the entire FAISS index to use the
joint model — you can't mix CLIP embeddings with OpenAI text embeddings
in one index without dimension mismatches.

Caption-then-embed sidesteps all of that: the vision model runs ONCE at
ingest time, produces a text description, and everything downstream
(embedding, chunking, FAISS) is identical to what Phase 4 already built.
The cost is that visual detail the captioning model misses is genuinely
lost — but for BI chart retrieval, a GPT-4o-mini caption of a revenue
trend chart captures everythi

## 10. Caption prompt contract

The prompt asks for **150–250 words** covering:

1. Visual type
2. Axis/column meaning and units
3. Primary business finding/trend
4. Anomalies, highlights, or thresholds
5. Time period / dataset scope

The prompt explicitly says not to invent unreadable numbers.

This matters because the caption is not merely explanatory prose: it becomes the **retrieval representation of the image**.


In [7]:
# Demonstrate the expected image conversion contract without calling the API.
try:
    from agentic_bi.rag.multimodal.captioner import image_to_base64

    # Replace with a real image path to execute:
    SAMPLE_IMAGE = PROJECT_ROOT / "documents/multimodal/charts/01_monthly_revenue_trend.png"

    if SAMPLE_IMAGE:
        b64_data, media_type = image_to_base64(Path(SAMPLE_IMAGE))
        print("base64 type :", type(b64_data).__name__)
        print("base64 chars:", len(b64_data))
        print("media_type  :", media_type)
        print("data URI prefix:", f"data:{media_type};base64,"[:50])
    else:
        print("Set SAMPLE_IMAGE to a real PNG/JPG/WEBP path to inspect the conversion.")
except Exception as e:
    print("Project import unavailable:", type(e).__name__, "-", e)


base64 type : str
base64 chars: 158988
media_type  : image/png
data URI prefix: data:image/png;base64,


## 11. Optional `extra_context`: why `key_insight` is passed to the model

For chart/table images, metadata may contain a known `key_insight`.

The captioner appends it to the prompt as additional context:

```text
Image
  +
known key_insight
  │
  ▼
Vision LLM
  │
  ▼
caption
```

The purpose is to help the model verify important business facts that may be difficult to read from a small image.

If captioning fails, `caption_image_safe()` can return the provided fallback instead of aborting the whole batch.


# Part III — Orchestration

## 12. `ingest.py`: the pipeline controller

The public function is:

```python
ingest_multimodal_corpus(
    multimodal_dir=None,
    max_caption_workers=4,
    force_recaption=False
) -> FAISS
```

Its four major stages are:

1. **Load corpus**
2. **Caption images**
3. **Filter unusable content**
4. **Embed and save FAISS**

The function returns the in-memory `FAISS` object after also persisting it to disk.


In [8]:
ingest_source = Path(SRC_DIR / "agentic_bi/rag/multimodal/ingest.py").read_text()
print(ingest_source[:8500])


"""
ingest.py
─────────
Orchestrates the complete multimodal ingestion pipeline:

    load corpus
        │
        ▼
    caption images          ← GPT-4o-mini vision, one API call per image
        │
        ▼
    build Documents         ← page_content = caption text (+ PDF text)
        │
        ▼
    embed with              ← text-embedding-3-small (same as text RAG)
    text-embedding-3-small
        │
        ▼
    save FAISS index        ← data/vectorstore/faiss_multimodal/

Key design decisions
────────────────────
1.  Separate FAISS index.
    The multimodal corpus is stored in faiss_multimodal/, distinct from
    the text-only faiss_index/.  This lets both indices exist side-by-side
    so the retrieval layer can query either or both, and means running
    multimodal ingestion does not invalidate text retrieval.

2.  Caption cache.
    API calls are expensive. A JSON sidecar file (caption_cache.json)
    stores file_path → caption so re-running ingestion only calls the
    vi

## 13. Caption cache — avoiding repeated vision calls

The cache is stored as:

```text
<index_dir>/caption_cache.json
```

The cache key is:

```text
file_path::mtime
```

Therefore:

- same path + same modification time → cache hit
- same path + changed modification time → new caption
- `force_recaption=True` → cache is ignored

This is an important production optimization because the vision call is the expensive/slow external operation in this ingestion path.


In [9]:
# Understand the cache key without requiring the full pipeline.
import tempfile, time

try:
    from agentic_bi.rag.multimodal.ingest import _cache_key

    with tempfile.NamedTemporaryFile(suffix=".png") as f:
        key = _cache_key(f.name)
        print("Example cache key:")
        print(key)
        print("\nStructure:")
        print("file_path :: modification_time")
except Exception as e:
    print("Project import unavailable:", type(e).__name__, "-", e)


Example cache key:
/var/folders/1w/k2xkpg3s50n1nnp_jrdp_pdw0000gn/T/tmpxdkdy2h_.png::1789663770

Structure:
file_path :: modification_time


## 14. Parallel captioning

The implementation uses:

```python
ThreadPoolExecutor(max_workers=max_workers)
```

Each image caption is an independent HTTP request, so the work can be parallelized.

The default is:

```python
MAX_CAPTION_WORKERS = 4
```

The result list is deliberately restored to the original document order using the original index associated with each future.

```text
docs_needing_caption
        │
        ├── future #0 ──┐
        ├── future #1 ──┤
        ├── future #2 ──┤──► results[index] = updated Document
        └── future #3 ──┘
```

If one caption fails, the code logs the error and keeps the original document, whose `page_content` may still contain the metadata `key_insight` fallback.


## 15. The `Document` changes during captioning

Before:

```python
Document(
    page_content=key_insight_or_empty,
    metadata={"needs_caption": True, ...}
)
```

After a successful caption:

```python
Document(
    page_content="<150–250 word visual description>",
    metadata={"needs_caption": True, ...}
)
```

Notice that `needs_caption` is not changed to `False` in `_caption_one()`. The flag is primarily used to decide whether the document enters the captioning branch; the final embedding decision is based on whether `page_content` contains usable text.


## 16. Final content-quality filter

After text-ready documents and captioned documents are combined:

```python
all_ready = text_ready + captioned
```

the pipeline keeps only documents satisfying:

```python
d.page_content and len(d.page_content.strip()) >= 20
```

So the final contract is:

```text
all_ready
   │
   ▼
content exists?
   │
   ▼
at least 20 non-whitespace characters?
   │
   ▼
embeddable
```

This protects the embedding stage from empty placeholders and extremely short content.


# Part IV — Embeddings and FAISS

## 17. Embedding stage

The pipeline calls:

```python
embeddings = get_embeddings()
vectorstore = FAISS.from_documents(docs, embeddings)
```

The documented design uses `text-embedding-3-small`.

At this point, the system no longer cares whether the original source was:

- a chart,
- a table,
- PDF prose, or
- an embedded PDF image.

Everything has been normalized into text + metadata.


## 18. What is actually stored conceptually?

For every final `Document`:

```text
Document
├── page_content
│     └── text used to create the embedding
│
└── metadata
      ├── source_type
      ├── file_path
      ├── file_name
      ├── title
      ├── time_period
      ├── data_source
      ├── page_number / image_index where relevant
      └── other source-specific fields
```

FAISS primarily handles the vector similarity side. The LangChain FAISS wrapper also retains the documents/metadata needed to map retrieved vectors back to their source context.


## 19. Separate multimodal FAISS index

The implementation deliberately saves the multimodal index separately:

```text
data/vectorstore/
├── faiss_index/              # existing text RAG index
└── faiss_multimodal/         # this pipeline
```

This avoids invalidating the existing text index and allows the retrieval layer to query either index or both later.

The multimodal index directory also contains the caption cache.


In [10]:
# Inspect the index directory if it already exists.
INDEX_DIR = PROJECT_ROOT / "data" / "vectorstore" / "faiss_multimodal"

if INDEX_DIR.exists():
    print("Index directory:", INDEX_DIR)
    for p in sorted(INDEX_DIR.iterdir()):
        print(" -", p.name)
else:
    print("No existing multimodal index found at:", INDEX_DIR)


No existing multimodal index found at: /Users/azizulshaikh/Projects/agentic_bi_platform/data/vectorstore/faiss_multimodal


# Part V — Full execution

## 20. Execute the complete ingestion pipeline

This is the notebook equivalent of the CLI call:

```bash
python scripts/ingest_multimodal.py
```

Equivalent Python:

```python
vectorstore = ingest_multimodal_corpus(...)
```

Parameters:

- `multimodal_dir`: corpus root
- `max_caption_workers`: number of parallel caption requests
- `force_recaption`: bypass caption cache

**Run this only when your project environment, corpus, embedding configuration, and vision-model credentials are available.**


In [11]:
# Full pipeline — execution cell.
RUN_FULL_INGESTION = False

if RUN_FULL_INGESTION:
    from agentic_bi.rag.multimodal.ingest import ingest_multimodal_corpus

    vectorstore = ingest_multimodal_corpus(
        multimodal_dir=CORPUS_DIR,
        max_caption_workers=4,
        force_recaption=False
    )
    print("Vectorstore type:", type(vectorstore).__name__)
else:
    vectorstore = None
    print("Set RUN_FULL_INGESTION = True to execute the real pipeline.")


Set RUN_FULL_INGESTION = True to execute the real pipeline.


## 21. Inspect the final vector store

Once ingestion has run, the most useful validation is to inspect retrieved `Document` objects.

Expected retrieval contract:

```python
results = vectorstore.similarity_search(query, k=3)
```

Output:

```python
list[Document]
```

Each result should contain:

```python
result.page_content
result.metadata
```

The `source_type` and `file_name` fields make it possible to explain where the retrieved information came from.


In [12]:
# Retrieval smoke test — mirrors ingest_multimodal.py.
QUERY = "return rate by product category"

if vectorstore is not None:
    results = vectorstore.similarity_search(QUERY, k=3)

    for i, doc in enumerate(results, 1):
        print(f"\n[{i}]")
        print("source_type:", doc.metadata.get("source_type"))
        print("file_name  :", doc.metadata.get("file_name"))
        print("title      :", doc.metadata.get("title"))
        print("content    :", doc.page_content[:500].strip(), "...")
else:
    print("Run the full ingestion cell first.")


Run the full ingestion cell first.


# Part VI — CLI layer

## 22. `ingest_multimodal.py`

The fourth file is intentionally thin. It does not implement the ingestion logic itself.

Its job is to:

1. parse CLI arguments,
2. configure logging,
3. call `ingest_multimodal_corpus()`,
4. optionally perform a dry run,
5. run a quick retrieval smoke test.

Important CLI controls:

```text
--corpus-dir
--workers
--force-recaption
--dry-run
--log-level
```

The dry-run path calls the loader only, so it can report corpus composition without making vision API calls.


In [13]:
cli_source = Path(PROJECT_ROOT / "scripts/ingest_multimodal.py").read_text()
print(cli_source[:6500])


"""
ingest_multimodal.py
────────────────────
CLI entry point for multimodal RAG ingestion.

Usage
─────
    # Standard run (uses caption cache)
    python scripts/ingest_multimodal.py

    # Force re-caption everything (e.g. after changing the prompt)
    python scripts/ingest_multimodal.py --force-recaption

    # Custom corpus directory
    python scripts/ingest_multimodal.py --corpus-dir /path/to/multimodal

    # Reduce parallelism if hitting rate limits
    python scripts/ingest_multimodal.py --workers 2

    # Dry-run: show what would be ingested without calling the API
    python scripts/ingest_multimodal.py --dry-run

Expected output
───────────────
    === Multimodal RAG Ingestion ===
    Corpus dir : documents/multimodal
    Index dir  : data/vectorstore/faiss_multimodal
    Loaded 10 chart documents
    Loaded 5 table documents
    Loaded 31 PDF documents (text + embedded images)
    Documents: 23 to caption, 23 text-ready
    Captioning complete: 0 cache hits, 23 new API c

## 23. Dry-run mental model

When `--dry-run` is used:

```text
CLI
 │
 ▼
_dry_run()
 │
 ▼
load_multimodal_corpus()
 │
 ▼
group by source_type
 │
 ├── chart
 ├── table
 ├── pdf_text
 └── pdf_image
 │
 ▼
report documents needing captioning
```

No vision call is made and no FAISS index is built.

This is useful for validating the corpus before spending API calls.


# Part VII — End-to-end variable map

## 24. Important variables and their input/output formats

| Variable | Type | Produced by | Consumed by | Meaning |
|---|---|---|---|---|
| `multimodal_dir` | `Path` | caller/settings | loader | Root corpus |
| `charts_meta` / `tables_meta` / `pdfs_meta` | `dict` | JSON loader | document loaders | Filename → metadata |
| `all_docs` | `list[Document]` | `load_multimodal_corpus()` | ingestion | Raw normalized corpus |
| `Document.page_content` | `str` | loader/captioner | embedding | Text representation |
| `Document.metadata` | `dict` | loader | retrieval/index | Source/provenance data |
| `needs_caption` | `bool` | loader | `_should_caption()` | Whether visual captioning is needed |
| `to_caption` | `list[Document]` | ingestion | caption runner | Image docs requiring vision |
| `text_ready` | `list[Document]` | ingestion | final merge | Already textual docs |
| `cache` | `dict` | cache loader | captioning | Previously generated captions |
| `caption` | `str` | vision model | `Document.page_content` | Searchable image description |
| `embeddable` | `list[Document]` | ingestion | FAISS | Final usable documents |
| `embeddings` | embedding object | `get_embeddings()` | FAISS | Text embedding model |
| `vectorstore` | `FAISS` | `_build_and_save()` | retrieval | Persistent vector index |
| `results` | `list[Document]` | `similarity_search()` | application | Retrieved evidence |

The central transformation is:

```text
heterogeneous files
      ↓
list[Document]
      ↓
image Documents get captions
      ↓
all Documents contain searchable text
      ↓
filtered list[Document]
      ↓
embeddings
      ↓
FAISS
      ↓
retrieved list[Document]
```


# 25. Worked example: one chart

Suppose the source is:

```text
charts/01_monthly_revenue_trend.png
```

and metadata contains:

```json
{
  "title": "Monthly Revenue Trend",
  "key_insight": "Revenue increased steadily during the observed period.",
  "chart_type": "line chart",
  "time_period": "2025",
  "data_source": "synthetic"
}
```

### Step 1 — Loader

```python
Document(
    page_content="Revenue increased steadily during the observed period.",
    metadata={
        "source_type": "chart",
        "file_name": "01_monthly_revenue_trend.png",
        "needs_caption": True,
        ...
    }
)
```

### Step 2 — Captioner

The image plus `key_insight` is sent to the vision model.

Output:

```python
caption: str
```

For example, conceptually:

```text
The visual is a line chart showing monthly revenue...
```

### Step 3 — Document update

```python
doc.page_content = caption
```

The metadata remains attached.

### Step 4 — Embedding

```python
caption → text embedding vector
```

### Step 5 — FAISS

The vector and its associated document are stored.

### Step 6 — Retrieval

A query such as:

```text
"How did revenue change over the year?"
```

can retrieve the chart because the visual information has been converted into semantically searchable text.


# 26. Worked example: PDF containing text + chart

A single PDF page can create multiple Documents.

```text
report.pdf
   │
   └── page 4
       ├── extracted prose
       │     └── Document(source_type="pdf_text",
       │                  needs_caption=False)
       │
       └── embedded chart
             └── Document(source_type="pdf_image",
                          needs_caption=True)
```

This is an important design choice: the prose and visual are independently retrievable.

The PDF image also carries:

```python
parent_pdf
page_number
image_index
```

so downstream systems can preserve provenance.


# 27. Error and fallback behavior

The pipeline has several defensive layers:

### Missing metadata

`_load_metadata_json()` returns `{}` and logs a warning.

### Missing/empty image directory

The image loader logs a warning and returns an empty list.

### PDF read failure

The individual PDF is logged and skipped rather than crashing the entire corpus load.

### Image caption failure

`caption_image_safe()` can return a fallback `key_insight`, allowing ingestion to continue.

### Empty final corpus

`_build_and_save()` raises:

```python
ValueError("No documents to index — corpus appears empty.")
```

### Missing existing FAISS index

`load_multimodal_vectorstore()` raises `FileNotFoundError` and tells the caller to ingest first.


# 28. Operational considerations

### Caption cache

Use the cache for normal runs. Use `force_recaption=True` after changing the caption prompt or vision model.

### Worker count

The default is 4. Lower it if the vision API rate limit becomes a bottleneck.

### Tiny embedded images

Images below 5 KB are ignored by `_should_caption()`.

### Separate index

Do not assume this index replaces the text-only FAISS index. The code intentionally keeps them separate.

### Security

`FAISS.load_local(..., allow_dangerous_deserialization=True)` is used by the existing implementation. Loading FAISS artifacts with dangerous deserialization enabled should therefore be restricted to indexes/artifacts you trust.


# 29. Final end-to-end checklist

Before considering multimodal ingestion complete, verify:

- [ ] `documents/multimodal/charts/` contains expected images.
- [ ] `documents/multimodal/tables/` contains expected images.
- [ ] `documents/multimodal/pdfs/` contains expected PDFs.
- [ ] Metadata JSON files exist where expected.
- [ ] `load_multimodal_corpus()` returns `list[Document]`.
- [ ] `source_type` correctly identifies each source.
- [ ] Image documents have `needs_caption=True`.
- [ ] PDF text documents have `needs_caption=False`.
- [ ] Caption cache is being reused on subsequent runs.
- [ ] Caption failures do not unnecessarily abort the complete batch.
- [ ] Short/empty documents are removed before embedding.
- [ ] The multimodal FAISS index is saved under `faiss_multimodal/`.
- [ ] `load_multimodal_vectorstore()` can reload it.
- [ ] A semantic retrieval smoke test returns sensible `Document` objects.
- [ ] Retrieved metadata is sufficient to trace results back to the original file/page/image.

## The one-line mental model

> **Load heterogeneous artefacts → normalize into LangChain Documents → caption only visual Documents → preserve metadata → filter unusable text → embed all text representations → persist a dedicated FAISS index → retrieve Documents with provenance.**
